In [1]:
import rqdatac as rq
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import RFECV
from sklearn.metrics import r2_score, mean_absolute_error

# Log in to rqdatac
rq.init()

# Get 沪深300 constituent stocks
constituents = rq.index_components('000300.XSHG')

start_date = '2010-01-01'
end_date = '2025-12-31'

factors = [
    'basic_earnings_per_share',
    'net_profit',
    'net_profitTTM',
    'net_profit_parent_company',
    'ebit',
    'ebitda',
    'return_on_equity',
    'return_on_asset'

    # revenue
    'revenue',
    'revenue_growth_rate',
    'net_profit_growth_rate',
    
    # valuation
    'book_value_per_share',
    'book_value_per_share_ttm',

    # working capital
    'working_capital',                     # Net liquidity
    'working_capital_ttm',                 # TTM working capital
    'inventory',                           # Inventory levels
    'net_accts_receivable',                # Receivables collection
    'current_assets',                      # Total current assets
    'current_liabilities',                 # Total current liabilities
    'cash_equivalent',                     # Cash position

    # investment and fixed assets
    'net_fixed_assets',                    # Asset base for earnings
    'depreciation_and_amortization',       # Non-cash expense affecting net income

    # balance sheet core metrics
    'total_assets',                        # Size basis
    'total_equity',                        # Book value base
    'equity_parent_company',               # Parent equity
    'intangible_assets',                   # Goodwill/intangibles (earnings quality)

    # liabilites structure
    'short_term_loans',                    # Short-term debt
    'long_term_loans',                     # Long-term debt
    'bond_payable',                        # Bond obligations
    'long_term_liabilities_due_one_year',  # Current portion of long-term debt

    # leverage/liquidity
    'debt_to_equity',
    'current_ratio',
]

target = 'basic_earnings_per_share'

# Fetch financial statement ratio information
financial_data = rq.get_factor(constituents, factors, start_date=start_date, end_date=end_date)

financial_data = financial_data.fillna(0, inplace=False)

print("financial data:")
print(financial_data.head(5))

financial data:
                          basic_earnings_per_share    net_profit  \
order_book_id date                                                 
688111.XSHG   2019-11-18                       0.0  2.044113e+08   
              2019-11-19                       0.0  2.044113e+08   
              2019-11-20                       0.0  2.044113e+08   
              2019-11-21                       0.0  2.044113e+08   
              2019-11-22                       0.0  2.044113e+08   

                          net_profitTTM  net_profit_parent_company  ebit  \
order_book_id date                                                         
688111.XSHG   2019-11-18            0.0               2.044113e+08   0.0   
              2019-11-19            0.0               2.044113e+08   0.0   
              2019-11-20            0.0               2.044113e+08   0.0   
              2019-11-21            0.0               2.044113e+08   0.0   
              2019-11-22            0.0            

In [2]:
X = financial_data

X = X.groupby([
    pd.Grouper(level='order_book_id'),
    pd.Grouper(level='date', freq='Y')
]).last().fillna(0)

X = X.reorder_levels(['order_book_id', 'date'])

print(X.head(5))

                          basic_earnings_per_share    net_profit  \
order_book_id date                                                 
000001.XSHE   2010-12-31                      1.46  4.733940e+09   
              2011-12-31                      2.01  7.745077e+09   
              2012-12-31                      2.00  1.034597e+10   
              2013-12-31                      1.43  1.169600e+10   
              2014-12-31                      1.37  1.569400e+10   

                          net_profitTTM  net_profit_parent_company  ebit  \
order_book_id date                                                         
000001.XSHE   2010-12-31   6.127250e+09               4.733940e+09   0.0   
              2011-12-31   9.310681e+09               7.687182e+09   0.0   
              2012-12-31   1.299139e+10               1.023789e+10   0.0   
              2013-12-31   1.486078e+10               1.169600e+10   0.0   
              2014-12-31   1.922900e+10               1.569400e+10 

In [3]:
# Fetch next 5 years earnings
future_earnings = rq.get_factor(constituents, target, start_date='2015-01-01', end_date=end_date)

future_earnings = future_earnings.groupby([
    pd.Grouper(level='order_book_id'),
    pd.Grouper(level='date', freq='Y')
]).last().fillna(0)

future_earnings = future_earnings.reorder_levels(['order_book_id', 'date'])

future_earnings = future_earnings.fillna(0, inplace=False)

y = future_earnings

print(future_earnings)

                          basic_earnings_per_share
order_book_id date                                
000001.XSHE   2015-12-31                      1.27
              2016-12-31                      1.09
              2017-12-31                      1.06
              2018-12-31                      1.14
              2019-12-31                      1.32
...                                            ...
688981.XSHG   2021-12-31                      0.93
              2022-12-31                      1.19
              2023-12-31                      0.46
              2024-12-31                      0.34
              2025-12-31                      0.48

[2927 rows x 1 columns]


In [4]:
# Assuming your data is already multi-indexed with (order_book_id, date)
# Align features (year t) with target (year t+1)
import numpy as np

# Method 1: Shift per stock
def align_features_target(X, y):
    """
    X: features at time t
    y: target at time t
    Returns: X_aligned (time t), y_aligned (time t+1)
    """
    # Reset index to work with dates
    X_df = X.reset_index()
    y_df = y.reset_index()
    
    # Rename target column for clarity
    y_df = y_df.rename(columns={'free_cash_flow_company_per_share_ttm': 'target'})
    
    # Create a shifted version of y for each stock
    aligned_pairs = []
    
    for stock in X_df['order_book_id'].unique():
        X_stock = X_df[X_df['order_book_id'] == stock].copy()
        y_stock = y_df[y_df['order_book_id'] == stock].copy()
        
        # Sort by date
        X_stock = X_stock.sort_values('date')
        y_stock = y_stock.sort_values('date')
        
        # Shift y backwards by 1 year (so y at t-1 pairs with X at t)
        # Or more intuitively: align X at year t with y at year t+1
        y_stock['target_next_year'] = y_stock['target'].shift(-1)
        
        # Merge on date
        merged = X_stock.merge(
            y_stock[['order_book_id', 'date', 'target_next_year']], 
            on=['order_book_id', 'date']
        )
        
        # Drop rows where next year's target is missing
        merged = merged.dropna(subset=['target_next_year'])
        
        aligned_pairs.append(merged)
    
    # Combine all stocks
    aligned_df = pd.concat(aligned_pairs, ignore_index=True)
    
    # Separate X and y
    X_aligned = aligned_df.drop(['order_book_id', 'date', 'target_next_year'], axis=1)
    y_aligned = aligned_df['target_next_year']
    
    return X_aligned, y_aligned, aligned_df[['order_book_id', 'date']]  # Keep metadata

def robust_align_features_target(X, y):
    """
    Robust alignment handling index mismatches
    """
    # Reset indices to columns
    X_df = X.reset_index()
    y_df = y.reset_index()
    
    # Rename target column
    y_df = y_df.rename(columns={'free_cash_flow_company_per_share_ttm': 'target'})
    
    # Sort both by stock and date
    X_df = X_df.sort_values(['order_book_id', 'date'])
    y_df = y_df.sort_values(['order_book_id', 'date'])
    
    aligned_pairs = []
    
    for stock in X_df['order_book_id'].unique():
        X_stock = X_df[X_df['order_book_id'] == stock].copy()
        y_stock = y_df[y_df['order_book_id'] == stock].copy()
        
        if len(y_stock) == 0:
            print(f"Warning: No target data for {stock}")
            continue
        
        # Ensure both have consistent date ranges
        common_dates = set(X_stock['date']).intersection(set(y_stock['date']))
        if len(common_dates) == 0:
            print(f"Warning: No common dates for {stock}")
            continue
        
        X_stock = X_stock[X_stock['date'].isin(common_dates)]
        y_stock = y_stock[y_stock['date'].isin(common_dates)]
        
        # Sort by date
        X_stock = X_stock.sort_values('date')
        y_stock = y_stock.sort_values('date')
        
        # Shift target to next year
        y_stock['target_next_year'] = y_stock['target'].shift(-1)
        
        # Merge on date
        merged = X_stock.merge(
            y_stock[['order_book_id', 'date', 'target_next_year']],
            on=['order_book_id', 'date'],
            how='inner'
        )
        
        # Drop rows with NaN target (last year of each stock)
        merged = merged.dropna(subset=['target_next_year'])
        
        if len(merged) > 0:
            aligned_pairs.append(merged)
    
    if len(aligned_pairs) == 0:
        raise ValueError("No aligned data found. Check your date ranges.")
    
    # Combine all stocks
    aligned_df = pd.concat(aligned_pairs, ignore_index=True)
    
    # Separate X, y, and metadata
    feature_cols = [col for col in X_df.columns if col not in ['order_book_id', 'date']]
    X_aligned = aligned_df[feature_cols]
    y_aligned = aligned_df['target_next_year']
    metadata = aligned_df[['order_book_id', 'date']]
    
    return X_aligned, y_aligned, metadata

WINDOW_IN = 5
WINDOW_OUT = 5

def create_5y_sequences(X, y):

    X_df = X.reset_index().sort_values(
        ['order_book_id', 'date']
    )

    y_df = y.reset_index().sort_values(
        ['order_book_id', 'date']
    )

    target_col = y_df.columns[-1]

    X_seq = []
    y_seq = []

    feature_cols = [
        c for c in X_df.columns
        if c not in ['order_book_id', 'date']
    ]

    for stock in X_df['order_book_id'].unique():

        xf = X_df[X_df['order_book_id'] == stock]
        yf = y_df[y_df['order_book_id'] == stock]

        xvals = xf[feature_cols].values
        yvals = yf[target_col].values

        if len(xvals) < WINDOW_IN + WINDOW_OUT:
            continue

        max_start = min(
            len(xvals),
            len(yvals)
        ) - WINDOW_IN - WINDOW_OUT + 1

        for i in range(max_start):

            X_seq.append(
                xvals[i:i + WINDOW_IN].flatten()
            )

            y_seq.append(
                yvals[
                    i + WINDOW_IN :
                    i + WINDOW_IN + WINDOW_OUT
                ]
            )

    # print("Number of samples:", len(y_seq))

    # for i in range(10):
    #     print(i, np.array(y_seq[i]).shape)
    
    # unique_shapes = set()

    # for item in y_seq:
    #     unique_shapes.add(np.array(item).shape)

    # print(unique_shapes)
    
    return np.array(X_seq), np.array(y_seq)

# Apply the robust alignment
try:
    # X_aligned, y_aligned, metadata = robust_align_features_target(X, y)
    X_aligned, y_aligned = create_5y_sequences(X, y)
    print(f"Success! Aligned {len(X_aligned)} samples")
    print(f"X shape: {X_aligned.shape}")
    print(f"y shape: {y_aligned.shape}")
    # print(f"Date range: {metadata['date'].min()} to {metadata['date'].max()}")
except ValueError as e:
    print(f"Alignment failed: {e}")

# # Apply alignment
# X_aligned, y_aligned, metadata = align_features_target(X, y)

# print(f"Original X shape: {X.shape}")
# print(f"Original y shape: {y.shape}")
# print(f"Aligned X shape: {X_aligned.shape}")
# print(f"Aligned y shape: {y_aligned.shape}")

Success! Aligned 447 samples
X shape: (447, 155)
y shape: (447, 5)


In [5]:
print(X_aligned)

[[1.46000000e+00 4.73394000e+09 6.12725000e+09 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [2.01000000e+00 7.74507700e+09 9.31068100e+09 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [2.98000000e-01 3.82564566e+09 6.58567124e+09 ... 0.00000000e+00
  0.00000000e+00 1.33921900e+00]
 ...
 [1.86000000e+00 1.45928383e+08 1.81915616e+08 ... 0.00000000e+00
  0.00000000e+00 1.10088970e+01]
 [1.90000000e-01 9.19305374e+08 1.15550068e+09 ... 0.00000000e+00
  0.00000000e+00 1.48745100e+00]
 [1.80000000e-01 8.32706167e+08 9.29776469e+08 ... 0.00000000e+00
  0.00000000e+00 2.92697000e+00]]


In [6]:
print(y_aligned)

[[ 1.11    1.4     1.78    1.94    1.94  ]
 [ 1.4     1.78    1.94    1.94    1.87  ]
 [ 1.741   1.436   1.4697  1.1585 -1.5132]
 ...
 [ 2.5     3.16    0.65    1.26    1.64  ]
 [ 0.075   0.165   0.248   0.114   0.39  ]
 [ 0.165   0.248   0.114   0.39    0.67  ]]


## 随机森林

In [7]:
# 随机森林模型

# Train test split 80/20
split = int(len(X_aligned) * 0.8)

X_train, X_test = X_aligned[:split], X_aligned[split:]
y_train, y_test = y_aligned[:split], y_aligned[split:]

# Define hyperparameters once
rf_params = dict(
    n_estimators=100,
    max_depth=3,
    min_samples_leaf=5,
    max_features='sqrt'
)

# Feature selection
model = RandomForestRegressor(**rf_params)
selector = RFECV(estimator=model, cv=5)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)  # apply same selection to test set

# Cross-validate, then final fit
rf_model = RandomForestRegressor(**rf_params)
cv_scores = cross_val_score(rf_model, X_train_selected, y_train, cv=5, scoring='r2')
print(f"Cross-validation R^2 scores: {cv_scores}")

# Fit the model
rf_model.fit(X_train_selected, y_train)

# Evaluate
y_pred = rf_model.predict(X_test_selected)
r2 = r2_score(y_test, y_pred, multioutput='uniform_average')
mae = mean_absolute_error(y_test, y_pred)
print(f"Test Average R^2: {r2}, Test MAE: {mae}")

r2_per_year = r2_score(
    y_test,
    y_pred,
    multioutput='raw_values'
)

print("Year+1 R^2:", r2_per_year[0])
print("Year+2 R^2:", r2_per_year[1])
print("Year+3 R^2:", r2_per_year[2])
print("Year+4: R^2", r2_per_year[3])
print("Year+5 R^2:", r2_per_year[4])

Cross-validation R^2 scores: [ 0.11951128 -0.07280909 -0.21254288  0.13913851  0.2479265 ]
Test Average R^2: 0.23258049700791011, Test MAE: 0.658837190166491
Year+1 R^2: 0.5455248193447787
Year+2 R^2: 0.10240451275482776
Year+3 R^2: -0.35516976312694504
Year+4: R^2 0.2912146471828696
Year+5 R^2: 0.5789282688840196


## Gradient Boosting Regression Model

In [8]:
# Grading Boosting Regression Model

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import cross_val_score
from sklearn.multioutput import MultiOutputRegressor

# Train-test split
split = int(len(X_aligned) * 0.8)

X_train, X_test = X_aligned[:split], X_aligned[split:]
y_train, y_test = y_aligned[:split], y_aligned[split:]

# Gradient Boosting Regressor model
gbr_model = MultiOutputRegressor(
    GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        random_state=42
    )
)

# Cross validation
gbr_cv_scores = cross_val_score(
    gbr_model,
    X_train,
    y_train,
    cv=5,
    scoring='r2'
)

print("Gradient Boosting Cross-validation R^2 scores:")
print(gbr_cv_scores)
print(f"Average CV R^2: {gbr_cv_scores.mean()}")

# Train model
gbr_model.fit(X_train, y_train)

# Prediction
gbr_pred = gbr_model.predict(X_test)

# Evaluation
gbr_r2 = r2_score(y_test, gbr_pred)
gbr_mae = mean_absolute_error(y_test, gbr_pred)

print(f"Gradient Boosting Test R^2: {gbr_r2}")
print(f"Gradient Boosting Test MAE: {gbr_mae}")

r2_per_year = r2_score(
    y_test,
    gbr_pred,
    multioutput='raw_values'
)

print("Year+1 R^2:", r2_per_year[0])
print("Year+2 R^2:", r2_per_year[1])
print("Year+3 R^2:", r2_per_year[2])
print("Year+4: R^2", r2_per_year[3])
print("Year+5 R^2:", r2_per_year[4])

Gradient Boosting Cross-validation R^2 scores:
[ 0.34146881  0.08077362 -0.20738251  0.19971953  0.28849206]
Average CV R^2: 0.14061430163415292
Gradient Boosting Test R^2: 0.043134201767323854
Gradient Boosting Test MAE: 0.7365538518644573
Year+1 R^2: 0.2878601272064075
Year+2 R^2: 0.15895211542432408
Year+3 R^2: -0.20371493246127104
Year+4: R^2 -0.09545513097552161
Year+5 R^2: 0.06802882964268031


## 长短期记忆

In [9]:
# 长短期记忆模型

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Train-test split
split = int(len(X_aligned) * 0.8)

X_train, X_test = X_aligned[:split], X_aligned[split:]
y_train, y_test = y_aligned[:split], y_aligned[split:]

# Scale features
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled = x_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled = y_scaler.transform(y_test)

# Reshape for LSTM
# Shape: (samples, timesteps, features)
n_features = X_aligned.shape[1] // 5

X_train_lstm = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    5,
    n_features
)

X_test_lstm = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    5,
    n_features
)

# Build LSTM model
lstm_model = Sequential()

lstm_model.add(
    LSTM(
        units=64,
        return_sequences=False,
        input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])
    )
)

lstm_model.add(Dropout(0.2))

lstm_model.add(Dense(32, activation='relu'))

lstm_model.add(Dense(5))

# Compile
lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse'
)

# Train
history = lstm_model.fit(
    X_train_lstm,
    y_train_scaled,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Predict
lstm_pred_scaled = lstm_model.predict(X_test_lstm)

# Inverse transform
lstm_pred = y_scaler.inverse_transform(lstm_pred_scaled)

# Evaluation
lstm_r2 = r2_score(y_test, lstm_pred, multioutput='uniform_average')
lstm_mae = mean_absolute_error(y_test, lstm_pred)

print(f"LSTM Test Average R^2: {lstm_r2}")
print(f"LSTM Test MAE: {lstm_mae}")

r2_per_year = r2_score(
    y_test,
    lstm_pred,
    multioutput='raw_values'
)

print("Year+1 R^2:", r2_per_year[0])
print("Year+2 R^2:", r2_per_year[1])
print("Year+3 R^2:", r2_per_year[2])
print("Year+4: R^2", r2_per_year[3])
print("Year+5 R^2:", r2_per_year[4])

2026-06-08 15:52:35.193491: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Epoch 1/50
9/9 [==============================] - 4s 111ms/step - loss: 0.0100 - val_loss: 0.0027
Epoch 2/50
9/9 [==============================] - 0s 11ms/step - loss: 0.0066 - val_loss: 0.0012
Epoch 3/50
9/9 [==============================] - 0s 11ms/step - loss: 0.0053 - val_loss: 0.0012
Epoch 4/50
9/9 [==============================] - 0s 12ms/step - loss: 0.0048 - val_loss: 9.8999e-04
Epoch 5/50
9/9 [==============================] - 0s 12ms/step - loss: 0.0044 - val_loss: 0.0011
Epoch 6/50
9/9 [==============================] - 0s 13ms/step - loss: 0.0043 - val_loss: 0.0011
Epoch 7/50
9/9 [==============================] - 0s 11ms/step - loss: 0.0035 - val_loss: 0.0012
Epoch 8/50
9/9 [==============================] - 0s 11ms/step - loss: 0.0034 - val_loss: 0.0014
Epoch 9/50
9/9 [==============================] - 0s 11ms/step - loss: 0.0031 - val_loss: 0.0012
Epoch 10/50
9/9 [==============================] - 0s 11ms/step - loss: 0.0033 - val_loss: 0.0013
Epoch 11/50
9/9 [=======

## 预测

In [10]:
# 预测

# Fetch financial statement ratio information
ly_financial_df = rq.get_factor('601318.XSHG', factors, start_date='2022-01-01', end_date='2026-05-01')

ly_financial_df = ly_financial_df.fillna(0, inplace=False)

ly_financial_df = ly_financial_df.groupby([
    pd.Grouper(level='order_book_id'),
    pd.Grouper(level='date', freq='Y')
]).last().fillna(0)

ly_financial_df = ly_financial_df.reorder_levels(['order_book_id', 'date'])

ly_financial_df = ly_financial_df.fillna(0, inplace=False)

# Keep only latest 5 years
ly_financial_df = ly_financial_df.tail(5)

print(ly_financial_df.head(20))

                          basic_earnings_per_share    net_profit  \
order_book_id date                                                 
601318.XSHG   2022-12-31                      4.38  9.511500e+10   
              2023-12-31                      4.94  1.081110e+11   
              2024-12-31                      6.73  1.400540e+11   
              2025-12-31                      7.56  1.550670e+11   
              2026-12-31                      1.43  3.326300e+10   

                          net_profitTTM  net_profit_parent_company  ebit  \
order_book_id date                                                         
601318.XSHG   2022-12-31   1.193690e+11               7.646300e+10   0.0   
              2023-12-31   1.040180e+11               8.757500e+10   0.0   
              2024-12-31   1.412170e+11               1.191820e+11   0.0   
              2025-12-31   1.617460e+11               1.328560e+11   0.0   
              2026-12-31   1.564050e+11               2.502200e+10 

In [11]:
prediction_input = (
    ly_financial_df
    .values
    .flatten()
    .reshape(1, -1)
)

In [12]:
# Random Forest Prediction
rf_prediction = rf_model.predict(
    selector.transform(prediction_input)
)

# Gradient Boosting Regressor Prediction
gbr_prediction = gbr_model.predict(
    prediction_input
)

# LSTM Prediction

# Scale
ly_scaled = x_scaler.transform(prediction_input)

# Reshape
n_features = ly_scaled.shape[1] // 5
ly_lstm = ly_scaled.reshape(
    ly_scaled.shape[0],
    5,
    n_features
)

# Predict
lstm_prediction_scaled = lstm_model.predict(ly_lstm)

# Inverse scale
lstm_prediction = y_scaler.inverse_transform(
    lstm_prediction_scaled
)

print(f"Random Forest Prediction: {rf_prediction}")
print(f"Gradient Boosting Regressor Prediction: {gbr_prediction}")
print(f"LSTM Prediction: {lstm_prediction}")

1/1 [==============================] - 0s 27ms/step
Random Forest Prediction: [[ 6.29425287  7.23428537  8.6504337   9.93509844 10.31047216]]
Gradient Boosting Regressor Prediction: [[19.74622929 19.30521495 28.58670473 38.84386886 30.69292848]]
LSTM Prediction: [[ 5.820366   7.9753575  7.904638  10.469271  15.745414 ]]


## 模型结果

In [13]:
# =========================
# Model Comparison
# =========================

comparison_df = pd.DataFrame({
    'Model': ['Random Forest', 'Gradient Boosting Regression', 'LSTM'],
    'R2': [
        r2,
        gbr_r2,
        lstm_r2
    ],
    'MAE': [
        mae,
        gbr_mae,
        lstm_mae
    ]
})

print(comparison_df)

                          Model         R2       MAE
0                 Random Forest   0.232580  0.658837
1  Gradient Boosting Regression   0.043134  0.736554
2                          LSTM -56.742484  2.763012


## Export Models

In [14]:
import joblib
import pickle
from tensorflow.keras.models import save_model

# =========================
# Export Models
# =========================

# 1. Export Random Forest model and its selector
joblib.dump(rf_model, 'earnings_random_forest_model.pkl')
joblib.dump(selector, 'earnings_rf_selector.pkl')
print("✓ Random Forest model and selector saved")

# 2. Export Gradient Boosting model
joblib.dump(gbr_model, 'earnings_gradient_boosting_model.pkl')
print("✓ Gradient Boosting model saved")

# 3. Export LSTM model
lstm_model.save('earnings_lstm_model.h5')
print("✓ LSTM model saved")

# 4. Export scalers (needed for LSTM preprocessing)
joblib.dump(x_scaler, 'earnings_x_scaler.pkl')
joblib.dump(y_scaler, 'earnings_y_scaler.pkl')
print("✓ Scalers saved")

# 5. Save feature columns and configuration
config = {
    'window_in': 5,
    'window_out': 5,
    'n_features': n_features,
    'factors': factors if 'factors' in dir() else None
}

with open('earnings_model_config.pkl', 'wb') as f:
    pickle.dump(config, f)
print("✓ Model configuration saved")

print("\n✅ All models exported successfully!")

✓ Random Forest model and selector saved
✓ Gradient Boosting model saved
✓ LSTM model saved
✓ Scalers saved
✓ Model configuration saved

✅ All models exported successfully!


/Users/tylerpruitt/Desktop/量化投资交易/Quant Trading Desk/.venv/lib/python3.8/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
